# Part 1 — MultiModal RAG over a Read-Only Knowledge Base

In [ ]:
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
    text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))


## 1.1 Ingestion & Indexing\nRequires the real `Document.pdf`, `Document2_...pdf`, `Document3_...pdf` and `kb_manifest.json` from Canvas/Kaggle, placed under `data/raw/part1/`. Not yet run in this environment — pending those files.

In [ ]:
from src.part1.ingest import ingest_documents
from src.part1.index_store import HybridIndex

MANIFEST_PATH = "../data/raw/part1/kb_manifest.json"
PDF_DIR = "../data/raw/part1"
FIGURES_DIR = "../data/processed/part1/figures"

text_chunks, figures = ingest_documents(MANIFEST_PATH, PDF_DIR, FIGURES_DIR)
print("text chunks:", len(text_chunks), "figures:", len(figures))


In [ ]:
index = HybridIndex(persist_dir="../data/processed/part1/chroma_index")
index.build_text_index(text_chunks)
index.build_figure_index(figures)


## 1.2 Retrieval & Grounded Answers

In [ ]:
from src.part1.answer_engine import AnswerEngine
from src.part1.conversation import ConversationStore

engine = AnswerEngine(index)
store = ConversationStore()
conv = store.new_conversation("demo")
result = engine.answer(conv, "What was GDP growth in 2011?")
result


## 1.3 Conversation Handling — multi-turn demo

In [ ]:
conv2 = store.new_conversation("demo2")
print(engine.answer(conv2, "What was GDP growth in 2011?"))
print(engine.answer(conv2, "And in 2012?"))


## 1.4 Evaluation, Safety & Documentation\nGolden set of 25 questions (`eval/part1_golden_set.json`) is written once the real documents are available, and committed before any tuning, per the lab's reproducibility rule.

In [ ]:
# import json
# with open("../eval/part1_golden_set.json") as f:
#     golden_set = json.load(f)
# TODO: run golden_set through engine.answer, score correctness/status, Recall@5, citation accuracy


## 1.5 Kaggle Leaderboard submission

In [ ]:
# from src.part1.kaggle_submit import build_submission
# build_submission()
